In [ ]:
!pip install uv

In [ ]:
!uv pip install openai pandas arize arize-otel openinference-instrumentation-openai openai-agents

In [ ]:
!git clone https://github.com/rk-yen/customer-support-collab.git

# Secrets (Before Starting) ...
Add the following variables in the Secrets section:
* ARIZE_API_KEY
* ARIZE_SPACE_ID
* OPENAI_API_KEY

In [ ]:
from pathlib import Path
import sys

import pandas as pd
pd.set_option('max_colwidth', None)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "workshop_helpers").exists():
    candidates = [path for path in REPO_ROOT.iterdir() if path.is_dir() and (path / "workshop_helpers").exists()]
    if not candidates:
        raise FileNotFoundError("Could not find a repo folder containing workshop_helpers")
    REPO_ROOT = candidates[0]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop_helpers.backend import (
    TOOLS,
    hydrate_backend_from_dataset,
    run_support_agent_async,
    snapshot_backend,
)
from workshop_helpers.data import DATASET
from workshop_helpers.experiments import (
    VARIANT_BEHAVIORS,
    format_checklist_rows,
    prepare_experiment_bundle,
    production_readiness_checklist,
    run_experiment,
    select_cases,
    summarize_dataset,
)
from workshop_helpers.metrics import (
    compare_scores,
    pack_response_payload,
    score_routing_response,
)
from workshop_helpers.scenarios import run_context_agent, run_raw_llm


In [ ]:
from workshop_helpers.setup import setup_clients

env = setup_clients()
client = env["client"]
ARIZE_SPACE_ID = env["arize_space_id"]
ARIZE_API_KEY = env["arize_api_key"]

## Evaluator Prompts

The routing eval uses exact match, but the support evals use AI judges.
Those judge prompts are editable here in the notebook so you can inspect and tune them.


In [ ]:
JUDGE_PROMPTS = {
    "tone": (
        "You evaluate customer support responses for tone.\n\n"
        "Return exactly two lines in this format:\n"
        "LABEL: <Good|Acceptable|Poor>\n"
        "REASONING: <one sentence>\n\n"
        "GOOD: Warm, empathetic, and professional.\n"
        "ACCEPTABLE: Polite but generic or slightly flat.\n"
        "POOR: Cold, robotic, dismissive, or argumentative."
    ),
    "outcome_system": "You are a strict evaluator of support response correctness.",
    "outcome": (
        "Evaluate whether the response reaches the right substantive outcome for the case.\n\n"
        "Customer message:\n{user_input}\n\n"
        "Expected output:\n{ideal}\n\n"
        "Actual response:\n{actual}\n\n"
        "Return exactly two lines in this format:\n"
        "LABEL: <Good|Acceptable|Poor>\n"
        "REASONING: <one sentence>\n\n"
        "GOOD: Reaches the same core outcome as the expected output.\n"
        "ACCEPTABLE: Mostly correct but missing important specifics or completeness.\n"
        "POOR: Wrong outcome, misses the main point, or fails to address the user need."
    ),
    "workflow_system": "You are a strict evaluator of support workflow behavior.",
    "workflow": (
        "Evaluate whether the response matches the expected workflow behavior for this system variant.\n\n"
        "System variant: {variant_name}\n"
        "Expected variant behavior: {variant_behavior}\n"
        "Variant-specific expectation for this case: {variant_expectation}\n"
        "Case workflow expectation: {workflow_expectation}\n"
        "Missing info required: {missing_info_required}\n"
        "Action expected: {action_expected}\n"
        "Action type: {action_type}\n\n"
        "Customer message:\n{user_input}\n\n"
        "Actual response:\n{actual}\n\n"
        "Return exactly two lines in this format:\n"
        "LABEL: <Good|Acceptable|Poor>\n"
        "REASONING: <one sentence>\n\n"
        "GOOD: Clearly matches the expected workflow behavior for this variant.\n"
        "ACCEPTABLE: Partially matches but is missing specificity or discipline.\n"
        "POOR: Does not match the expected workflow behavior for this variant."
    ),
}

print("Editable AI judge prompts loaded:")

---
## Experiment Setup

We will benchmark each stage right after tuning it, instead of waiting until the end.
That makes the workshop loop tighter:

1. Update the stage prompt and relevant evaluator prompts.
2. Re-run the single-case demo and scorecard.
3. Run the dataset experiment for that stage.
4. Only then move on to the next capability level.


In [ ]:
LIMIT_N_CASES = None  # set to an integer like 8 for a faster run
EXPERIMENT_DATASET = select_cases(DATASET, limit_n=LIMIT_N_CASES)

library_summary = summarize_dataset(DATASET)
experiment_summary = summarize_dataset(EXPERIMENT_DATASET)
backend_snapshot = hydrate_backend_from_dataset(DATASET)

print(f"Scenario library: {library_summary['scenario_count']} total cases")
print(f"Experiment dataset: {experiment_summary['scenario_count']} cases")
print(f"  Categories: {', '.join(experiment_summary['categories'])}")
print(f"  Limit applied: {LIMIT_N_CASES}")
print()
print("Backend refreshed from the scenario library:")
for label, value in backend_snapshot.items():
    print(f"  {label}: {value}")

pd.DataFrame(EXPERIMENT_DATASET)[["scenario_id", "category"]]


In [ ]:
STAGE_RUNS = {}

def build_current_experiment_bundle():
    return prepare_experiment_bundle(
        client=client,
        arize_api_key=ARIZE_API_KEY,
        arize_space_id=ARIZE_SPACE_ID,
        dataset=DATASET,
        prompt_router=globals().get("PROMPT_ROUTER", ""),
        prompt_v1=globals().get("PROMPT_V1", ""),
        prompt_v2=globals().get("PROMPT_V2", ""),
        prompt_v3=globals().get("PROMPT_V3", ""),
        judge_prompts=JUDGE_PROMPTS,
        limit_n=LIMIT_N_CASES,
    )

def run_stage_experiment(variant_name: str, name_prefix: str):
    bundle = build_current_experiment_bundle()
    task_map = {
        "router": "task_router",
        "v1": "task_v1",
        "v2": "task_v2",
        "v3": "task_v3",
    }
    run = run_experiment(
        arize_client=bundle["client"],
        dataset_id=bundle["dataset_id"],
        name_prefix=name_prefix,
        task=bundle["tasks"][task_map[variant_name]],
        evaluators=bundle["build_evaluators"](variant_name),
    )
    STAGE_RUNS[variant_name] = run
    print(f"Dataset name: {bundle['dataset_name']}")
    print(f"Dataset ID:   {bundle['dataset_id']}")
    print(f"Rows:         {bundle['row_count']}")
    print(f"Created now:  {bundle['created']}")
    print()
    print(f"Experiment ID: {run['experiment'].id}")
    print(f"Rows evaluated: {len(run['results_df'])}")
    return run


---
## Step 1 — Routing Agent

Before we draft support replies, we can start with a smaller AI task:
route the inbound message into one of the known support categories.

This is a good first eval because the target label already exists in the dataset
and we can score it with exact match instead of an AI judge.


In [ ]:
ROUTING_DEMO_CASE = {
    "scenario_id": "CS_015",
    "category": "returns",
    "user_input": "I received the wrong colour. I ordered the navy blue tote but got black instead.",
}

ROUTING_CATEGORIES = sorted({case["category"] for case in DATASET})
PROMPT_ROUTER = (
    "You are a support routing agent. "
    "Read the customer message and return exactly one category from this list: {categories}. "
    "Return only the category label and nothing else."
).format(categories=", ".join(ROUTING_CATEGORIES))

router_output = run_raw_llm(client, ROUTING_DEMO_CASE["user_input"], PROMPT_ROUTER).strip()
router_payload = pack_response_payload(router_output)

print("Routing categories:")
print(", ".join(ROUTING_CATEGORIES))
print()
print("Router prediction:")
print("-" * 50)
print(router_output)
print("-" * 50)
print(f"Expected category: {ROUTING_DEMO_CASE['category']}")


### Scoring — exact match

For routing, we can use a simple 'exact match' evaluator.

| Variant | What we score |
|--------|----------------|
| `router` | `exact_match` |

If the predicted category matches the dataset category, it passes.


In [ ]:
routing_scorecard = pd.DataFrame(
    [
        {
            "variant": "router",
            **score_routing_response(router_payload, ROUTING_DEMO_CASE["category"]),
        }
    ]
)

display(routing_scorecard[["variant", "predicted_category", "expected_category", "exact_match", "total"]])
display(routing_scorecard[["exact_match_reasoning"]])


## Update `PROMPT_ROUTER`

Adjust `PROMPT_ROUTER`, rerun the router demo cells above, and confirm the exact-match score looks solid.
Once the demo case is behaving well, run the dataset benchmark for the routing stage.


In [ ]:
run_router = run_stage_experiment("router", "router-category-classifier")
# df_router = run_router["results_df"]
# display(df_router.head())


---
## Step 2 — Raw LLM Call

The simplest possible baseline: a two-sentence system prompt, no customer data.
This is what most teams' v0 looks like — pasting into ChatGPT.

In [ ]:
PROMPT_V1 = "You are a customer support agent. Help the customer."
CUSTOMER_MESSAGE = (
    "I received the wrong colour. I ordered the navy blue tote but got black instead."
)

DEMO_CASE = {
    "scenario_id": "CS_015",
    "category": "returns",
    "user_input": CUSTOMER_MESSAGE,
    "source_data": {
        "customer_name": "Jack Morrison",
        "customer_id": "CUST_5540",
        "order_id": "ORD_3318",
        "product_name": "Classic Canvas Tote - Navy Blue",
        "order_date": "2024-03-17",
        "days_since_order": 5,
        "order_total": 32.0,
        "order_status": "delivered",
        "return_policy_days": 30,
        "account_status": "active",
        "notes": "Warehouse pick error confirmed in fulfilment log. Navy Blue variant in stock.",
    },
    "expected_output": "Hi Jack, I'm really sorry about this - that's a fulfilment error on our side and I apologise for the inconvenience. I'll ship the correct Navy Blue Classic Canvas Tote to you today at no extra charge. I'll also include a prepaid return label for the black one, though there's no rush on sending it back. You should receive dispatch details within a few hours.",
    "missing_info_required": False,
    "workflow_expectation": "perform_action",
    "action_expected": True,
    "action_type": "send_replacement",
}

output_v1 = run_raw_llm(client, CUSTOMER_MESSAGE, PROMPT_V1)

print("V1 (raw LLM):")
print("-" * 50)
print(output_v1)
print("-" * 50)


### Scoring — variant-aware metrics

| Variant | What we score |
|--------|----------------|
| `v1` | `tone_quality`, `workflow_fit` |

`v1` is not judged on full outcome correctness because it does not have enough information to fully resolve many cases.

In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "v1": output_v1,
        },
        case=DEMO_CASE,
        variant_behaviors=VARIANT_BEHAVIORS,
        judge_prompts=JUDGE_PROMPTS,
    )
)

display(scorecard[["variant", "workflow_fit", "tone", "total"]])
display(scorecard[["variant", "workflow_reasoning", "tone_reasoning"]])

**What is missing?**  The model has no idea who the customer is, what they ordered,
or what your return policy says. It can only ask for more information — or worse,
make something up.

## Update PROMPT_V1
Use https://platform.openai.com/chat/edit?models=gpt-4o-mini&optimize=true and update `PROMPT_V1`

### Objective
_Improve this baseline customer support system prompt for cases where the model only has the customer's message and no backend access. The response should be empathetic, concise, and honest about uncertainty. It must not invent order details, policy details, dates, or actions taken. When information is missing, it should ask only the single most useful follow-up question. Optimize for better tone and clearer issue handling, while keeping the limitations of a raw prompt-only assistant explicit._

### Benchmark `v1`

After updating `PROMPT_V1` and any relevant entries in `JUDGE_PROMPTS`, rerun the `v1` demo cells above.
When the single-case scorecard looks good enough, run the dataset benchmark for `v1`.


In [ ]:
run_v1 = run_stage_experiment("v1", "v1-raw-llm")
# df_v1 = run_v1["results_df"]
# display(df_v1.head())


---
## Step 3 — Add Context

Inject the customer's account data directly into the prompt.
No tools yet — just better information.
This is the **prompted agent** pattern.

In [ ]:
import json

def build_context_message(customer_message: str, source_data: dict) -> str:
    context_block = json.dumps(source_data, indent=2)
    return f"Customer context (internal):\n{context_block}\n\nCustomer message:\n{customer_message}"

SOURCE_DATA = DEMO_CASE["source_data"]
PROMPT_V2 = (
    "You are a customer support agent looking at an internal support snapshot. "
    "Use the provided context carefully and be specific. "
    "Do not claim that you already issued a refund, sent a label, escalated a case, or made any backend change. "
    "If the snapshot is enough to explain the likely next step, do that clearly. "
    "If a backend action would still need to happen, describe the next step without pretending it is already done."
)

print("Context passed to the model:")
customer_message_with_context = build_context_message(CUSTOMER_MESSAGE, SOURCE_DATA)
print(customer_message_with_context)
print()

output_v2 = run_context_agent(client, customer_message_with_context, PROMPT_V2)

print("V2 (with context):")
print("-" * 50)
print(output_v2)
print("-" * 50)


### Scoring — variant-aware metrics

| Variant | What we score |
|--------|----------------|
| `v1` | `tone_quality`, `workflow_fit` |
| `v2` | `tone_quality`, `workflow_fit`, `correct_outcome` |

`v1` is not judged on full outcome correctness because it does not have enough information to fully resolve many cases.

`v2` should use static context to tell the user the right next step.

In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "v1": output_v1,
            "v2": output_v2,
        },
        case=DEMO_CASE,
        variant_behaviors=VARIANT_BEHAVIORS,
        judge_prompts=JUDGE_PROMPTS,
    )
)

display(scorecard[["variant", "correct_outcome", "workflow_fit", "tone", "total"]])
display(scorecard[["variant", "outcome_reasoning", "workflow_reasoning", "tone_reasoning"]])

## Update `PROMPT_V2` and Evaluators

Adjust `PROMPT_V2` and any relevant judge prompts in `JUDGE_PROMPTS`, then rerun the `v2` demo cells above.
Once the single-case comparison looks strong, run the `v2` benchmark before moving to tools.


In [ ]:
run_v2 = run_stage_experiment("v2", "v2-context-agent")
# df_v2 = run_v2["results_df"]
# display(df_v2.head())


---
## Step 4 — Tool-Calling Agent (OpenAI Agents SDK)

V2 trusts whatever we inject. If `days_since_order` is stale in the database,
the model uses the wrong value. Policy arithmetic ("is this within 30 days?") is
just math — a model should not guess at it.

**Tools** let the model call real functions: it decides *what* to look up,
and *when* to act. That is the difference between a prompted assistant and an agent.

We use the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/),
which handles the tool-calling loop, tracing, and function schema generation
from Python type hints and docstrings.

```
  Customer message
       |
       v
  [ gpt-4o-mini ]  -- get_customer_profile(CUST_6601) -->  ORDER_DB + ACCOUNT_DB
       |                                                          |
       |          <-- profile: orders, subscription, app --------+
       |
       +-- check_return_eligibility(ORD_8814) --> 35 days, outside window by 5
       |
       v
  Declines + offers supervisor escalation
```

In [ ]:
backend_snapshot = snapshot_backend()

print("Simulated systems ready:")
for label, value in backend_snapshot.items():
    print(f"  {label}: {value}")


In [ ]:
print("Tools available to the agent:")
for tool in TOOLS:
    print(f"  - {tool.name}")


In [ ]:
AUTHENTICATED_CUSTOMER_ID = SOURCE_DATA["customer_id"]
PROMPT_V3 = (
    "You are a customer support agent for an online retail store. "
    "The authenticated customer ID for this session is: {authenticated_customer_id}. "
    "Always call get_customer_profile first to understand the customer before responding. "
    "Use get_order_details for shipping, billing, warranty, and fulfillment issues tied to an order. "
    "Use check_return_eligibility before approving or declining any return. "
    "If the tools give you enough information to complete the request, you must take the action before replying. "
    "Use the explicit action tools when an action is needed: issue_refund, send_return_label, send_replacement, escalate, send_unlock_email, resend_reset_email, cancel_subscription, or apply_subscription_credit. "
    "Do not merely promise an action in prose when a tool can perform it. "
    "Mention the confirmed action result in the response. "
    "Be empathetic and specific. Never invent figures or dates."
)

agent_result = await run_support_agent_async(
    customer_message=CUSTOMER_MESSAGE,
    instructions=PROMPT_V3.format(authenticated_customer_id=AUTHENTICATED_CUSTOMER_ID),
)
output_v3_text = agent_result.final_output
tool_calls = [
    {
        "name": raw.name,
        "arguments": getattr(raw, "arguments", ""),
    }
    for item in agent_result.new_items
    for raw in [getattr(item, "raw_item", None)]
    if raw and hasattr(raw, "name")
]
action_calls = [call for call in tool_calls if call["name"] in {
    "issue_refund",
    "send_return_label",
    "send_replacement",
    "escalate",
    "send_unlock_email",
    "resend_reset_email",
    "cancel_subscription",
    "apply_subscription_credit",
}]
output_v3 = pack_response_payload(
    output_v3_text,
    tool_calls=tool_calls,
    action_calls=action_calls,
)

print("PROMPT_V3:")
print(PROMPT_V3)
print()
print("V3 (Agents SDK):")
print("-" * 50)
print(output_v3_text)
print("-" * 50)
print()
print("Tool call chain:")
for call in tool_calls:
    print(f"  -> {call['name']}({call['arguments']})")


### Scoring — variant-aware metrics

| Variant | What we score |
|--------|----------------|
| `v1` | `tone_quality`, `workflow_fit` |
| `v2` | `tone_quality`, `workflow_fit`, `correct_outcome` |
| `v3` | `tone_quality`, `workflow_fit`, `correct_outcome`, `action_execution` |

`v1` is not judged on full outcome correctness because it does not have enough information to fully resolve many cases.

`v2` should use static context to tell the user the right next step.

`v3` should use tools to complete the action when enough information exists, and the evaluator now checks recorded action calls instead of only relying on final-response wording.


In [ ]:
scorecard = pd.DataFrame(
    compare_scores(
        client,
        {
            "v1": output_v1,
            "v2": output_v2,
            "v3": output_v3,
        },
        case=DEMO_CASE,
        variant_behaviors=VARIANT_BEHAVIORS,
        judge_prompts=JUDGE_PROMPTS,
    )
)

display(scorecard[["variant", "correct_outcome", "workflow_fit", "tone", "action_execution", "total"]])
display(scorecard[["variant", "outcome_reasoning", "workflow_reasoning", "tone_reasoning", "action_reasoning"]])

## Update `PROMPT_V3` and Evaluators

Adjust `PROMPT_V3` and any relevant judge prompts in `JUDGE_PROMPTS`, then rerun the `v3` demo cells above.
When the single-case scorecard and tool behavior look right, run the `v3` benchmark.


In [ ]:
run_v3 = run_stage_experiment("v3", "v3-tool-agent")
# df_v3 = run_v3["results_df"]
# display(df_v3.head())


---
## Step 5 — Production Readiness

Use the stage-by-stage benchmark results in Arize to fill in this checklist.
The goal is not a perfect score. The goal is to know:
- where prompting alone is enough to improve the experience
- where static context is enough to answer well
- where only tools create the right customer experience
- which workflows are still missing the backend support needed for a real agent


In [ ]:
for row in format_checklist_rows(production_readiness_checklist()):
    print(row)
